# Manhattan Ride-Sharing Simulator

Interactive notebook to visualize agents moving around Manhattan using Plotly animations.

**Controls:** Use the Play/Pause buttons to control the animation. You can also zoom and pan the map.

In [1]:
# Import the simulator and node selection function
from simulator import ManhattanSimulator
from run_simulator import select_three_fixed_nodes
from IPython.display import HTML, display

def show_plotly_figure(fig):
    """Display a Plotly figure directly in notebook without connection issues."""
    html = fig.to_html(include_plotlyjs='cdn', full_html=False)
    display(HTML(html))

## 1. Initialize the Simulator

This loads the Manhattan road network and sets up traffic parameters.

In [2]:
# Configuration
SEED = 42
CYCLE_LENGTH = 90.0  # Traffic light cycle in seconds
RANDOM_OFFSETS = True  # Random traffic light offsets

# Initialize simulator
sim = ManhattanSimulator(
    place_name="Manhattan, New York City, New York, USA",
    zone_shp="../data/processed/taxi_zones.shp",
    no_congestion=False,
    cycle_length=CYCLE_LENGTH,
    random_offsets=RANDOM_OFFSETS,
    seed=SEED
)

print(f"\nGraph loaded with {len(sim.all_nodes)} nodes")

Loading Manhattan road network...
Computing shortest path distances...
Building traffic light parameters...
Loaded graph with 2088 nodes and 4262 edges

Graph loaded with 2088 nodes


## 2. Run Multi-Agent Simulation (Shortest Path Policy)

Run a simulation with multiple agents using optimal transport matching.

In [3]:
# Simulation parameters
NUM_AGENTS = 10
MAX_TIME = 3600.0  # Maximum simulation time (seconds)
DT = 0.5  # Time step (seconds)

# ============================================
# ROUND TRIP MODE
# ============================================
# If True, agents travel back to start after reaching pickup
ROUND_TRIP = True

# ============================================
# FIXED NODE SELECTION (match your training!)
# ============================================
# Set to True to use 3 fixed nodes (same as --sample_starts_from_three_fixed in training)
USE_FIXED_STARTS = False  # Change to True to match training
USE_FIXED_PICKUPS = False  # Change to True if you used --sample_pickups_from_three_fixed

# Selection method: "random", "degree", "closeness", "betweenness"
# MUST match the --three_fixed_selection_method used in training!
SELECTION_METHOD = "betweenness"

# Select fixed nodes if enabled
fixed_start_nodes = None
fixed_pickup_nodes = None

if USE_FIXED_STARTS:
    print(f"Selecting 3 fixed START nodes (method: {SELECTION_METHOD}, seed: {SEED})...")
    fixed_start_nodes = select_three_fixed_nodes(
        G=sim.G,
        node_to_idx=sim.node_to_idx,
        seed=SEED,
        selection_method=SELECTION_METHOD,
        node_to_zone=sim.node_to_zone,
        target="starts"
    )
    print(f"Fixed start nodes: {fixed_start_nodes}\n")

if USE_FIXED_PICKUPS:
    print(f"Selecting 3 fixed PICKUP nodes (method: {SELECTION_METHOD}, seed: {SEED})...")
    fixed_pickup_nodes = select_three_fixed_nodes(
        G=sim.G,
        node_to_idx=sim.node_to_idx,
        seed=SEED,
        selection_method=SELECTION_METHOD,
        node_to_zone=sim.node_to_zone,
        target="pickups"
    )
    print(f"Fixed pickup nodes: {fixed_pickup_nodes}\n")

# Run simulation with shortest path policy
agents, assignments = sim.run_simulation(
    num_agents=NUM_AGENTS,
    num_pickups=NUM_AGENTS,  # One pickup per agent
    max_time=MAX_TIME,
    dt=DT,
    seed=SEED,
    q_table_path=None,  # Use shortest path policy
    fixed_start_nodes=fixed_start_nodes,
    fixed_pickup_nodes=fixed_pickup_nodes,
    round_trip=ROUND_TRIP  # Agents return to start after reaching pickup
)

print(f"\nSimulation complete!")
print(f"Assignments: {len(assignments)} agents assigned to pickups")

Using shortest path policy

Assignments (Optimal Transport, Shortest Path Policy):
  Agent 0: Node 42437753 → Node 42445617 (SP: 12 nodes, 197.8s)
  Agent 1: Node 42439203 → Node 42448707 (SP: 8 nodes, 145.8s)
  Agent 2: Node 42455026 → Node 42428223 (SP: 21 nodes, 321.6s)
  Agent 3: Node 42430320 → Node 42428272 (SP: 2 nodes, 29.3s)
  Agent 4: Node 42432191 → Node 42430603 (SP: 31 nodes, 373.9s)
  Agent 5: Node 42452816 → Node 42438060 (SP: 12 nodes, 157.4s)
  Agent 6: Node 9906966855 → Node 4161568888 (SP: 15 nodes, 361.7s)
  Agent 7: Node 42452396 → Node 42445534 (SP: 6 nodes, 76.0s)
  Agent 8: Node 42440004 → Node 42430698 (SP: 8 nodes, 190.7s)
  Agent 9: Node 588546798 → Node 42431898 (SP: 11 nodes, 111.2s)

[t=187.5s] Reassignment #1 (1 agents):
  Agent 3: Node 42430320 → Node 42446668 (14 nodes, 248.2s)

[t=325.5s] Reassignment #2 (1 agents):
  Agent 9: Node 588546798 → Node 1773084407 (62 nodes, 567.7s)

[t=456.0s] Reassignment #3 (1 agents):
  Agent 7: Node 42452396 → Node 424

In [4]:
# Create and display the animation
fig = sim.create_multi_agent_animation(
    agents=agents,
    assignments=assignments,
    fps=30,
    show_paths=True,
    title=f"Manhattan Simulation ({NUM_AGENTS} agents, Shortest Path)"
)

# Display directly in notebook (no connection required!)
show_plotly_figure(fig)

## 3. Run with Q-Table Greedy Policy (Optional)

Load a pre-trained Q-table and run the simulation using the greedy policy.

In [5]:
# Path to your Q-table (update this path as needed)
Q_TABLE_PATH = "q_tables/qlearning_4zones_100kepochs_20agents_seed0_random_offsets_3starts_betweenness_90cycle_1dt_eps1_plusreturn.pkl"

import os
if os.path.exists(Q_TABLE_PATH):
    print(f"Q-table found: {Q_TABLE_PATH}")
    
    # Run simulation with Q-table policy
    # Uses the same fixed_start_nodes, fixed_pickup_nodes, and round_trip from above!
    agents_q, assignments_q = sim.run_simulation(
        num_agents=NUM_AGENTS,
        num_pickups=NUM_AGENTS,
        max_time=MAX_TIME,
        dt=DT,
        seed=SEED,
        q_table_path=Q_TABLE_PATH,
        fixed_start_nodes=fixed_start_nodes,
        fixed_pickup_nodes=fixed_pickup_nodes,
        round_trip=ROUND_TRIP
    )
    
    # Create animation
    fig_q = sim.create_multi_agent_animation(
        agents=agents_q,
        assignments=assignments_q,
        fps=30,
        show_paths=True,
        title=f"Manhattan Simulation ({NUM_AGENTS} agents, Q-Table Greedy)"
    )
    show_plotly_figure(fig_q)
else:
    print(f"Q-table not found at: {Q_TABLE_PATH}")
    print("Update the Q_TABLE_PATH variable above if you have a trained Q-table.")

Q-table found: q_tables/qlearning_4zones_100kepochs_20agents_seed0_random_offsets_3starts_betweenness_90cycle_1dt_eps1_plusreturn.pkl
Loaded Q-table from q_tables/qlearning_4zones_100kepochs_20agents_seed0_random_offsets_3starts_betweenness_90cycle_1dt_eps1_plusreturn.pkl
  Shape: (249, 249, 90, 3)
  dt: 1.0
  Max time slices: 90
  Q-table policy may not work correctly!
Using Q-table greedy policy


IndexError: index 860 is out of bounds for axis 0 with size 249

## 4. Save Animation to HTML (Optional)

Save any figure to an HTML file for sharing or viewing later.

In [ ]:
# Save the animation to HTML file
OUTPUT_FILE = "simulation_from_notebook.html"

sim.save_html(fig, OUTPUT_FILE)
print(f"Animation saved to: {OUTPUT_FILE}")

Animation saved to simulation_from_notebook.html
Animation saved to: simulation_from_notebook.html
